# Lecture 1: Sampling distributions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture01/lecture01.ipynb)

Applied Quantitative Methods, Fall 2026. Readings: Newbold, Carlson and Thorne, sections 6.1 to 6.4 and 7.1.

This notebook follows the lecture slides. It starts with the case, RetailCorp, and the question for the day.
Each of the four parts after that ends with an exercise. In an exercise you
predict a result, run the cell, change one value, and answer a question about what you see. You do not need to
write any code. The last section holds all the calculations from the slides, so you can use the notebook as a
reference afterwards.

Run the setup cell first. It loads the RetailCorp data from the course repository.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
sample = pd.read_csv(url + "orders_sample.csv")          # our random sample of 200 orders
population = pd.read_csv(url + "orders_population.csv")  # all 50,000 orders

print(f"Sample: {len(sample)} orders. Population: {len(population):,} orders.")
sample.head()

## The case and the question

RetailCorp is a fictional Danish online retailer of home and kitchen goods. Management is considering a promise
to customers that 95% of orders arrive within 48 hours. Before deciding, it wants to know three things:

| Question | Statistic | Part |
|---|---|---|
| How long does a delivery take on average? | x̄, the sample mean of delivery times | 2 |
| What share of orders arrives late? | p̂, the sample proportion of orders taking more than 48 hours | 3 |
| How predictable are delivery times? | s², the sample variance of delivery times | 4 |

Part 1 comes first and asks what makes such a formula a good estimator.

The company and its data are simulated, and we use them in lectures 1 to 4. In practice we only ever have the
sample. We also have the population here because we simulated it, which lets us check how well sample results
work.

A parameter describes the population and is fixed. A statistic is computed from the sample and changes from
sample to sample. For the population we divide by N when we compute the standard deviation (`ddof=0`). For the
sample we divide by n − 1 (`ddof=1`, which is the default in pandas).


In [ ]:
# Population parameters
mu = population["delivery_hours"].mean()
sigma = population["delivery_hours"].std(ddof=0)
P = population["late"].mean()

# Sample statistics
x_bar = sample["delivery_hours"].mean()
s = sample["delivery_hours"].std()          # pandas divides by n - 1
p_hat = sample["late"].mean()

print(f"Population: mu = {mu:.2f} hours, sigma = {sigma:.2f} hours, P = {P:.4f}")
print(f"Sample:     x_bar = {x_bar:.2f} hours, s = {s:.2f} hours, p_hat = {p_hat:.3f}")

### Every sample gives a different result

If RetailCorp drew another random sample of 200 orders, how different would the results be? A simple random
sample gives every order the same chance of being selected, and with a list of all orders the computer draws the
sample for us. The cell below draws the five extra samples on the slide and prints x̄, s and p̂ for each.


In [ ]:
rng = np.random.default_rng(5)
print(f"Our sample: x_bar = {x_bar:.2f} hours, s = {s:.2f} hours, p_hat = {p_hat:.3f}")
for i in range(5):
    rows = rng.choice(len(population), size=200, replace=False)   # positions of 200 random orders
    new = population.iloc[rows]
    print(f"Sample {i + 2}:   x_bar = {new['delivery_hours'].mean():.2f} hours, "
          f"s = {new['delivery_hours'].std():.2f} hours, p_hat = {new['late'].mean():.3f}")


**Test yourself.** The next cell has no fixed seed, so every run draws a new random sample of 200 orders. Run it
a few times and compare the results with the cards on the slide or with your neighbor's screen. How far from
μ = 36.40, σ = 7.82 and P = 0.0695 do the statistics land?


In [ ]:
rng = np.random.default_rng()   # no seed: a new sample every time
rows = rng.choice(len(population), size=200, replace=False)
new = population.iloc[rows]
print(f"x_bar = {new['delivery_hours'].mean():.2f} hours, s = {new['delivery_hours'].std():.2f} hours, "
      f"p_hat = {new['late'].mean():.3f}")


## Part 1: Judging an estimator

A statistic changes from sample to sample, so a formula has to be judged over all possible samples. An
estimator is a rule for computing an estimate of a parameter from a sample, such as x̄ for μ and s² for σ².

- **Unbiased:** over many samples, its average equals the parameter.
- **Efficient:** among unbiased estimators, it has the smallest variance.
- **Consistent:** it gets closer to the parameter as n grows.


### Dividing by n or by n − 1

The deviations from the mean always sum to zero, so once n − 1 of them are known, the last one is fixed. pandas
divides by n − 1 by default, numpy divides by n unless you set `ddof=1`.

In [ ]:
x = pd.Series([2, 4, 9])
print("Deviations from the mean:", list(x - x.mean()), " sum:", (x - x.mean()).sum())
print("pandas x.var()         (divides by n - 1):", round(x.var(), 2))
print("numpy np.var(x)        (divides by n):    ", round(np.var(x), 2))
print("numpy np.var(x, ddof=1)(divides by n - 1):", round(np.var(x, ddof=1), 2))

### Exercise 1: dividing by n or by n − 1

**Predict.** The cell draws 5,000 samples of n = 5 orders and computes the variance of delivery times in two
ways, dividing by n and dividing by n − 1. Which average will be closer to the population variance? Write down
your guess.

**Run** the cell.

In [ ]:
n = 5            # sample size; change this to 50 in step 3
reps = 5000      # number of samples
rng = np.random.default_rng(1)
pop = population["delivery_hours"].to_numpy()

var_n, var_n1 = [], []
for _ in range(reps):
    x = rng.choice(pop, size=n, replace=False)
    var_n.append(x.var(ddof=0))     # divide by n
    var_n1.append(x.var(ddof=1))    # divide by n - 1

print(f"Population variance:         {pop.var():.1f}")
print(f"Average, dividing by n:      {np.mean(var_n):.1f}")
print(f"Average, dividing by n - 1:  {np.mean(var_n1):.1f}")

**Change** `n = 5` to `n = 50` and run the cell again.

**Answer.** What happens to the gap between the two averages, and what does that say about consistency?

## Part 2: The sampling distribution of the mean

The sampling distribution of x̄ has center μ and spread σ/√n, the standard error. If the population is normal,
x̄ is normal: x̄ ~ N(μ, σ²/n). As in Newbold, the second number in N(μ, σ²) is the variance. In scipy, `scale` is
the standard deviation, so for x̄ we use `scale=sigma/np.sqrt(n)`.

In [ ]:
n = 200
se = sigma / np.sqrt(n)
print(f"Standard error of the mean for n = {n}: {se:.3f} hours")

# The sampling distribution of x-bar as a scipy distribution
xbar_dist = stats.norm(loc=mu, scale=se)
print(f"P(x_bar > 37.5) for n = {n}: {xbar_dist.sf(37.5):.4f}")

### The central limit theorem

If the population is not normal, x̄ is still approximately normal when n is large. Basket values are a good
test case, because most orders are a few hundred kroner and a few are several thousand.

In [ ]:
values = population["order_value_dkk"]
print(f"Mean {values.mean():.0f} DKK, median {values.median():.0f} DKK, skewness {stats.skew(values):.1f}")

plt.hist(values[values <= 3000], bins=np.arange(0, 3001, 100), color="#31A9C1", edgecolor="white")
plt.xlabel("Basket value (DKK), orders above 3,000 DKK not shown")
plt.ylabel("Number of orders")
plt.title("Basket values of all 50,000 orders")
plt.show()

### Exercise 2: sample means of a skewed variable

**Predict.** The population standard deviation of basket values is σ = 521 DKK. What will the standard deviation
of the sample means be for n = 25? And for n = 100?

**Run** the cell. It draws 2,000 samples of size n and plots their means.

In [ ]:
n = 25           # sample size; try 100, then 5
reps = 2000
rng = np.random.default_rng(2)
pop = population["order_value_dkk"].to_numpy()

means = []
for _ in range(reps):
    x = rng.choice(pop, size=n, replace=False)
    means.append(x.mean())
means = np.array(means)

print(f"Population standard deviation (sigma):     {pop.std():.0f} DKK")
print(f"Standard deviation of the {reps:,} sample means: {means.std():.0f} DKK")
print(f"sigma / sqrt(n):                           {pop.std() / np.sqrt(n):.0f} DKK")

plt.hist(means, bins=40, color="#31A9C1", edgecolor="white")
plt.xlabel("Sample mean of basket value (DKK)")
plt.ylabel("Number of samples")
plt.title(f"Means of {reps:,} samples of size n = {n}")
plt.show()

**Change** `n` to 100, then to 5, and look at the shape of the histogram.

**Answer.** For which n does the histogram look normal, and when is the skew still visible?

## Part 3: The sampling distribution of a proportion

The share of late orders is the mean of a 0/1 variable (late = 1), so p̂ has a sampling distribution like x̄.
Its center is P and its standard error is √(P(1 − P)/n). Newbold's rule of thumb: the normal approximation is
good if nP(1 − P) > 5.

In [ ]:
print(f"Late orders in our sample: {sample['late'].sum()} of {len(sample)}, p_hat = {p_hat:.3f}")
print(f"Population share late: P = {P:.4f}")

for n in (25, 200):
    se_p = np.sqrt(P * (1 - P) / n)
    print(f"n = {n:>3}: standard error = {se_p:.4f}, nP(1 - P) = {n * P * (1 - P):.1f}")

### Exercise 3: late orders in samples of 200

**Predict.** In the population, 6.95% of orders are late. In what share of samples of 200 orders will more than
10% be late?

**Run** the cell. It draws 2,000 samples of 200 orders.

**Compare** the standard deviation of the 2,000 sample proportions with √(P(1 − P)/n).

In [ ]:
n = 200
reps = 2000
rng = np.random.default_rng(3)
pop = population["late"].to_numpy()

p_hats = []
for _ in range(reps):
    x = rng.choice(pop, size=n, replace=False)
    p_hats.append(x.mean())          # share of late orders in this sample
p_hats = np.array(p_hats)

print(f"Standard deviation of the {reps:,} sample proportions: {p_hats.std():.4f}")
print(f"sqrt(P(1 - P) / n):                             {np.sqrt(P * (1 - P) / n):.4f}")
print(f"Share of samples with more than 10% late:       {np.mean(p_hats > 0.10):.3f}")

plt.hist(p_hats, bins=np.arange(-0.0025, 0.16, 0.005), color="#31A9C1", edgecolor="white")
plt.axvline(0.10, color="#DF8E2E", lw=2)
plt.xlabel("Share of late orders in the sample")
plt.ylabel("Number of samples")
plt.title(f"Share late in {reps:,} samples of {n} orders")
plt.show()

## Part 4: The sampling distribution of the variance

If the population is normal, (n − 1)s²/σ² follows a chi-square distribution with n − 1 degrees of freedom. The
chi-square distribution takes only positive values, is skewed to the right, and has mean ν and variance 2ν,
where ν is the degrees of freedom.

In [ ]:
x = np.linspace(0.01, 45, 400)
for df in (4, 9, 24):
    plt.plot(x, stats.chi2.pdf(x, df), lw=2, label=f"{df} degrees of freedom")
    print(f"df = {df:>2}: mean = {stats.chi2.mean(df):.0f}, variance = {stats.chi2.var(df):.0f}")
plt.legend()
plt.title("Chi-square densities")
plt.show()

The chi-square result requires a normal population. Delivery times are close to normal, basket values are
strongly skewed.

In [ ]:
print(f"Skewness of delivery times: {stats.skew(population['delivery_hours']):.2f}")
print(f"Skewness of basket values:  {stats.skew(population['order_value_dkk']):.2f}")

### Exercise 4: chi-square and its assumption

**Predict.** The cell computes (n − 1)s²/σ² for 2,000 samples of 25 delivery times and draws the chi-square
density with 24 degrees of freedom on top. Will the histogram match the curve?

**Run** the cell.

In [ ]:
variable = "delivery_hours"   # change to "order_value_dkk" in step 3
n = 25
reps = 2000
rng = np.random.default_rng(4)
pop = population[variable].to_numpy()
sigma2 = pop.var()

stat = []
for _ in range(reps):
    x = rng.choice(pop, size=n, replace=False)
    stat.append((n - 1) * x.var(ddof=1) / sigma2)
stat = np.array(stat)

lo, hi = stats.chi2.ppf([0.025, 0.975], df=n - 1)
outside = np.mean((stat < lo) | (stat > hi))
print(f"Share of samples outside the middle 95% of the chi-square distribution: {outside:.1%}")

x = np.linspace(0.01, 80, 400)
plt.hist(stat, bins=np.arange(0, 81, 2), density=True, color="#31A9C1", edgecolor="white")
plt.plot(x, stats.chi2.pdf(x, df=n - 1), color="#211A52", lw=2, label=f"chi-square, {n - 1} df")
plt.xlabel("(n - 1) s² / σ²")
plt.legend()
plt.title(variable)
plt.show()

**Change** `variable` to `"order_value_dkk"` and run the cell again.

**Answer.** Where does the histogram differ from the curve, and what does that mean for probabilities computed
with the chi-square distribution?

## Calculations from the slides

The cells below repeat every calculation from the slides in the order they appear. They use `mu`, `sigma`, `P`,
`x_bar` and `s` from Part 1, so run the setup cell and the first code cell of "The case and the question" before these.

### Probability for a sample mean

A weekly audit tracks 25 random orders. With μ = 36.40 and σ = 7.82 hours, how likely is an audit mean above
39 hours?

In [ ]:
n = 25
se = sigma / np.sqrt(n)
z = (39 - mu) / se
print(f"Standard error: {se:.3f} hours")
print(f"z = {z:.2f}")
print(f"P(x_bar > 39) = {stats.norm.sf(z):.3f}")

# The same in one step: the sampling distribution of x-bar is normal with scale = se
print(f"One step:       {stats.norm.sf(39, loc=mu, scale=se):.3f}")

### Two kinds of spread

Single delivery times vary with standard deviation σ. Means of samples of 200 orders vary with the standard
error σ/√200.

In [ ]:
hours = population["delivery_hours"].to_numpy()
rng = np.random.default_rng(6)
means_200 = []
for _ in range(1000):
    x = rng.choice(hours, size=200, replace=False)
    means_200.append(x.mean())
means_200 = np.array(means_200)

print(f"Standard deviation of single deliveries in our sample: {s:.2f} hours")
print(f"Standard deviation of 1,000 sample means (n = 200):     {means_200.std():.2f} hours")
print(f"sigma / sqrt(200):                                      {sigma / np.sqrt(200):.2f} hours")

bins = np.arange(14, 63, 2)
fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
axes[0].hist(sample["delivery_hours"], bins=bins, color="#31A9C1", edgecolor="white")
axes[0].set_title("200 single deliveries in our sample")
axes[1].hist(means_200, bins=bins, color="#211A52", edgecolor="white")
axes[1].set_title("Means of 1,000 samples of 200 deliveries")
axes[1].set_xlabel("Delivery time in hours")
plt.tight_layout()
plt.show()

### The 48-hour promise

The promise that 95% of orders arrive within 48 hours concerns single orders, so we need the share of single
orders within 48 hours. We use x̄ and s from the sample in place of μ and σ.

In [ ]:
z = (48 - x_bar) / s
print(f"z = {z:.2f}")
print(f"Share within 48 hours, normal approximation: {stats.norm.cdf(z):.3f}")

# The same in one step: single delivery times, so scale = s
print(f"One step:                                     {stats.norm.cdf(48, loc=x_bar, scale=s):.3f}")

# Counting directly in the sample
print(f"Share within 48 hours in the sample:          {(sample['delivery_hours'] <= 48).mean():.3f}")

# Mean needed for 95% within 48 hours with the same spread
print(f"Mean needed for 95% within 48 hours:          {48 - stats.norm.ppf(0.95) * s:.1f} hours")

### Probability for a sample proportion

For samples of 200 orders with P = 0.0695, how likely is a sample with more than 10% late orders? The exact
binomial probability is shown for comparison: more than 10% means at least 21 late orders.

In [ ]:
n = 200
se_p = np.sqrt(P * (1 - P) / n)
z = (0.10 - P) / se_p
print(f"Standard error: {se_p:.4f}")
print(f"z = {z:.3f}")
print(f"P(p_hat > 0.10), normal approximation: {stats.norm.sf(z):.3f}")
print(f"P(X >= 21), exact binomial:             {stats.binom.sf(20, n, P):.3f}")

### Probability for a sample variance

For the weekly audit of 25 orders, with σ² = 61.14, how likely is a sample standard deviation above 10 hours?
s > 10 is the same as s² > 100.

In [ ]:
n = 25
sigma2 = sigma**2
chi2_value = (n - 1) * 100 / sigma2
print(f"sigma^2 = {sigma2:.2f}")
print(f"Chi-square value: {chi2_value:.2f} with {n - 1} degrees of freedom")
print(f"P(s > 10) = {stats.chi2.sf(chi2_value, df=n - 1):.3f}")

### Three sampling distributions

If RetailCorp drew another random sample, how different would the result be?

| Statistic | Center | Standard error | Shape | Holds when |
|---|---|---|---|---|
| x̄ (Part 2) | μ | σ/√n | Normal | The population is normal, or n is large (central limit theorem) |
| p̂ (Part 3) | P | √(P(1 − P)/n) | Approximately normal | nP(1 − P) > 5 |
| s² (Part 4) | σ² | Handled through the chi-square result | (n − 1)s²/σ² follows a chi-square distribution with n − 1 degrees of freedom | The population is normal. A large n does not help. |


### Symbols

| Symbol | Read as | Meaning |
|---|---|---|
| μ, σ, σ² | mu, sigma, sigma squared | Population mean, standard deviation and variance |
| x̄, s, s² | x-bar, s, s squared | Sample mean, standard deviation and variance |
| P, p̂ | P, p-hat | Population and sample proportion |
| N, n | N, n | Population size and sample size |
| θ, θ̂ | theta, theta-hat | Any parameter, and its estimator |
| E[ ] | expected value of | The average over all possible samples |
| ~ | is distributed as | For example, x̄ ~ N(μ, σ²/n) |
| σ_x̄, σ_p̂ | sigma x-bar, sigma p-hat | Standard error of the mean and of the proportion |
| χ², ν | chi-square, nu | The chi-square distribution and its degrees of freedom |

### Formulas and Python functions

| What | Formula | Python |
|---|---|---|
| Sample mean | x̄ = Σxᵢ/n | `df["col"].mean()` |
| Sample variance | s² = Σ(xᵢ − x̄)²/(n − 1) | `df["col"].var()` or `np.var(x, ddof=1)` |
| Standard error of x̄ | σ/√n | `sigma / np.sqrt(n)` |
| Probability for x̄ | Z = (x̄ − μ)/(σ/√n) | `stats.norm.sf(value, loc=mu, scale=sigma/np.sqrt(n))` |
| Standard error of p̂ | √(P(1 − P)/n) | `np.sqrt(P * (1 - P) / n)` |
| Probability for p̂ | Z = (p̂ − P)/σ_p̂ | `stats.norm.sf(value, loc=P, scale=se_p)` |
| Probability for s² | χ² = (n − 1)s²/σ², n − 1 df | `stats.chi2.sf(chi2_value, df=n - 1)` |

`stats.norm.cdf` gives the probability below a value, `stats.norm.sf` the probability above it, and
`stats.norm.ppf` the value with a given probability below it. The chi-square functions work the same way.

### Danish terms

| English | Danish |
|---|---|
| sample | stikprøve |
| estimator, point estimate | estimator, punktestimat |
| unbiased, efficient, consistent | middelret, effektiv, konsistent |
| standard deviation | standardafvigelse |
| standard error | standardfejl |
| sampling distribution | stikprøvefordeling |
| central limit theorem | den centrale grænseværdisætning |
| sample proportion | stikprøveandel |
| degrees of freedom | frihedsgrader |